In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from lib.helpers import get_run_date
from lib.job_control import get_max_insert_ts, insert_log
from pyspark.sql.functions import current_timestamp, lit, max as spark_max, row_number, coalesce
from pyspark.sql.window import Window
from lib.aws_s3 import warehouse_dir
from delta.tables import DeltaTable

In [ ]:
# Create the SparkSession
"""
spark = (
    SparkSession.builder
    .appName("14 - Part Production")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .enableHiveSupport()
    .getOrCreate()
)

spark
"""

In [ ]:
# Job parameters
now_ts = spark.range(1).select(current_timestamp().alias("now")).first()["now"]
run_date = get_run_date()
table = "part"
schema_name = "prd"
table_name = f"dim_{table}"
table_full_name = f"{schema_name}.{table_name}"
staging_table_full_name = f"stg.dim_{table}_stg"

print("****************************************")
print("14 - Part Production")
print("****************************************")
print(f"now_ts = {now_ts}")
print(f"run_date = {run_date}")
print(f"table = {table}")
print(f"schema_name = {schema_name}")
print(f"table_name = {table_name}")
print(f"table_full_name = {table_full_name}")
print(f"staging_table_full_name = {staging_table_full_name}")
print("")

In [ ]:
# Get max insert_ts for {schema_name}.{table_name} table from prd.job_control
max_insert_ts = get_max_insert_ts(spark, schema_name, table_name)
print(f"Max insert_ts in {table_full_name}: {max_insert_ts}\n")

In [ ]:
# Read data from staging
df_stg = (
    spark
    .table(staging_table_full_name)
)

In [ ]:
print(f"df_stg: {df_stg.count()} records read from {staging_table_full_name}")
df_stg.show()
df_stg.printSchema()

In [ ]:
df_dim_temp = (
    df_stg
    .withColumn("insert_ts", lit(now_ts))
    .withColumn("update_ts", lit(now_ts))
)

In [ ]:
print("Added audit columns")
print(f"df_dim_temp: {df_dim_temp.count()} records")
df_dim_temp.show()
df_dim_temp.printSchema()

In [ ]:
print(f"{table_full_name}: Before insert")
spark.table(f'{schema_name}.dim_{table}').show(truncate=False)

In [ ]:
# SCD1 update/insert to prd.dim_part

dt_dim = DeltaTable.forName(spark, table_full_name)

scd1_cols = ["part_name", "part_category"]

src = df_dim_temp.drop("part_key").where("part_code <> 'N/A'")   # one row per part_code

new_cols = src.columns + ["part_key"]

# Insert the unknown member (part_key = -1) if it doesn't exist
unknown_row = (
    spark.createDataFrame(
        [(-1, "N/A", "Not applicable", "Not applicable")],
        ["part_key", "part_code", "part_name", "part_category"],
    )
    .withColumn("insert_ts", lit(now_ts))
    .withColumn("update_ts", lit(now_ts))
)

(
    dt_dim.alias("t")
    .merge(unknown_row.alias("s"), "t.part_key = s.part_key")
    .whenNotMatchedInsertAll()
    .execute()
)

# Current highest surrogate key (0 if there are no real rows yet).
max_key = (
    spark.table(table_full_name)
    .where("part_key > 0")
    .agg(coalesce(spark_max("part_key"), lit(0)).alias("max_key"))
    .first()["max_key"]
)

# Window used to assign keys to new parts only
w = Window.orderBy("part_code")

# Find which incoming codes already exist in the target
cur = (
    spark.table(table_full_name)
    .selectExpr("part_code", "TRUE AS cur_exists")
)

flagged = src.join(cur, "part_code", "left")

# existing codes: null placeholder key, only so the union lines up.
row_a = (
    flagged.where("cur_exists IS NOT NULL")
           .withColumn("part_key", lit(None).cast("long"))
)

# new codes: contiguous keys are assigned here, and only here
row_b = (
    flagged.where("cur_exists IS NULL")
           .withColumn("part_key", (row_number().over(w) + lit(max_key)).cast("long"))
)

# Cache and materialize so both of Delta's passes see identical data
staged = row_a.unionByName(row_b).cache()
staged.count()

# Merge
(
    dt_dim.alias("t")
    .merge(staged.alias("s"), "t.part_code = s.part_code")

    # 1) existing parts: SCD1 overwrite if anything differs
    .whenMatchedUpdate(
        condition=" OR ".join(f"NOT (t.{c} <=> s.{c})" for c in scd1_cols),
        set={
            "part_name": "s.part_name",
            "part_category": "s.part_category",
            "update_ts": "s.update_ts",
        },
    )

    # 2) new part: insert
    .whenNotMatchedInsert(
        values={c: f"s.{c}" for c in new_cols},
    )
    .execute()
)

staged.unpersist()

In [ ]:
print(f"{table_full_name}: After insert (Impacted records)")
spark.table(f'{schema_name}.dim_{table}').where(f"update_ts = '{now_ts}'").show()

In [ ]:
# Add entry to prd.job_control
if insert_log(spark, schema_name, table_name, now_ts, run_date):
    print(f"{schema_name}.{table_name} entry added to prd.job_control")

In [ ]:
# Review added entry
spark.sql(f"select * from prd.job_control where table_name = '{table_name}' order by insert_ts desc limit 1").show(truncate=False)

In [ ]:
# Stop spark session
#spark.stop()